# TriangleMesh Example
This example demonstrates how to create a sphere, instantiate a TriangleMesh, check its edge manifold properties, and safely remove triangles from it.

In [1]:
from conquer3d.data_structure import TriangleMesh

import torch
import plotly.graph_objects as go
import conquer3d

### 1. Create a Sphere

In [2]:
vertices, faces = conquer3d.creation.create_sphere(sectors=32, stacks=16, radius=1.0)
print(f"Vertices: {vertices.shape}, Faces: {faces.shape}")

Vertices: torch.Size([482, 3]), Faces: torch.Size([960, 3])


### 2. Create Triangle Mesh and Check Manifold
A closed sphere should be a perfect closed manifold (every edge has exactly 2 triangles).

In [3]:
mesh: TriangleMesh = conquer3d.data_structure.TriangleMesh(vertices.to('cuda'), faces.to('cuda'))
is_manifold_open = mesh.is_edge_manifold(allow_boundary_edge=True)
is_manifold_closed = mesh.is_edge_manifold(allow_boundary_edge=False)

print(f"Is Open Manifold (<=2 triangles per edge): {is_manifold_open}")
print(f"Is Closed Manifold (==2 triangles per edge): {is_manifold_closed}")

is_vertex_manifold = mesh.is_vertex_manifold()
print(f"Is Vertex Manifold: {is_vertex_manifold}")

Is Open Manifold (<=2 triangles per edge): True
Is Closed Manifold (==2 triangles per edge): True
Is Vertex Manifold: True


### 3. Visualize Original Mesh (with Wireframe)

In [4]:
def plot_mesh(V_cpu, F_cpu, E_cpu, title):
    # Create wireframe lines by connecting edge vertices with None in between to break the line
    import numpy as np
    x_lines = []
    y_lines = []
    z_lines = []
    for edge in E_cpu:
        x_lines.extend([V_cpu[edge[0], 0], V_cpu[edge[1], 0], None])
        y_lines.extend([V_cpu[edge[0], 1], V_cpu[edge[1], 1], None])
        z_lines.extend([V_cpu[edge[0], 2], V_cpu[edge[1], 2], None])
        
    wireframe = go.Scatter3d(
        x=x_lines, y=y_lines, z=z_lines,
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False
    )
    
    mesh3d = go.Mesh3d(
        x=V_cpu[:, 0], y=V_cpu[:, 1], z=V_cpu[:, 2],
        i=F_cpu[:, 0], j=F_cpu[:, 1], k=F_cpu[:, 2],
        opacity=0.8,
        color='lightblue',
        flatshading=True,
        showscale=False
    )

    fig = go.Figure(data=[mesh3d, wireframe])
    fig.update_layout(title=title, scene=dict(aspectmode='data'))
    fig.show()

plot_mesh(vertices.numpy(), faces.numpy(), mesh.edges.cpu().numpy(), "Original Sphere")

### 4. Remove Random Faces
We create a boolean mask to randomly drop 20% of the triangles. This will create holes in the mesh!

In [5]:
# Create a random boolean mask keeping 80% of faces
torch.manual_seed(42)
keep_mask = torch.rand(mesh.num_triangles, device='cuda') > 0.2
mesh.remove_triangles_by_mask(keep_mask)

print(f"Remaining Faces: {mesh.num_triangles}")
is_manifold_open_new = mesh.is_edge_manifold(allow_boundary_edge=True)
is_manifold_closed_new = mesh.is_edge_manifold(allow_boundary_edge=False)

print(f"Is Open Manifold After Removal: {is_manifold_open_new}")
print(f"Is Closed Manifold After Removal: {is_manifold_closed_new}")

Remaining Faces: 771
Is Open Manifold After Removal: True
Is Closed Manifold After Removal: False


### 5. Visualize Decimated Mesh
Notice how the mesh is no longer a closed manifold because we introduced boundary edges (holes) by removing triangles.

In [6]:
plot_mesh(mesh.vertices.cpu().numpy(), mesh.triangles.cpu().numpy(), mesh.edges.cpu().numpy(), "Decimated Sphere")

### 4. Remove Random Triangles and Find Non-Manifold Vertices
We drop 20% of the triangles randomly. This will create open boundaries, but mathematically, random dropping on a dense sphere almost guarantees the creation of several 'bowtie' vertices (where two triangles touch at exactly one corner).
Let's find them and highlight them in red!

In [7]:
import torch
import random
torch.manual_seed(42)

# Drop ~20% of the triangles randomly
keep_mask = torch.rand(mesh.num_triangles, device='cuda') > 0.2

# Remove the triangles (this automatically clears all our C++ caches)
mesh.remove_triangles_by_mask(keep_mask)

print(f"Remaining triangles: {mesh.num_triangles}")
is_edge_manifold = mesh.is_edge_manifold(allow_boundary_edge=True)
print(f"Is Edge Manifold (Open): {is_edge_manifold}")

is_vertex_manifold = mesh.is_vertex_manifold()
print(f"Is Vertex Manifold: {is_vertex_manifold}")

# Detect the non-manifold (bowtie) vertices we just accidentally created!
non_manifold_vertices = mesh.get_non_manifold_vertices()
print(f"Number of Non-Manifold Vertices found: {len(non_manifold_vertices)}")


Remaining triangles: 627
Is Edge Manifold (Open): True
Is Vertex Manifold: False
Number of Non-Manifold Vertices found: 213


In [8]:
# Let's visualize the broken mesh and explicitly highlight the non-manifold vertices!
vertices_cpu = mesh.vertices.cpu().numpy()
triangles_cpu = mesh.triangles.cpu().numpy()

# Main mesh surface
mesh_plot = go.Mesh3d(
    x=vertices_cpu[:, 0],
    y=vertices_cpu[:, 1],
    z=vertices_cpu[:, 2],
    i=triangles_cpu[:, 0],
    j=triangles_cpu[:, 1],
    k=triangles_cpu[:, 2],
    color='lightblue',
    opacity=0.8,
    flatshading=True
)

# Wireframe of the remaining edges
edges = mesh.edges.cpu().numpy()
edge_x = []
edge_y = []
edge_z = []
for edge in edges:
    edge_x.extend([vertices_cpu[edge[0], 0], vertices_cpu[edge[1], 0], None])
    edge_y.extend([vertices_cpu[edge[0], 1], vertices_cpu[edge[1], 1], None])
    edge_z.extend([vertices_cpu[edge[0], 2], vertices_cpu[edge[1], 2], None])

wireframe = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='black', width=2),
    showlegend=False
)

# Non-manifold vertices highlighted in RED
nm_idx = non_manifold_vertices.cpu().numpy()
nm_points = go.Scatter3d(
    x=vertices_cpu[nm_idx, 0],
    y=vertices_cpu[nm_idx, 1],
    z=vertices_cpu[nm_idx, 2],
    mode='markers',
    marker=dict(color='red', size=6, symbol='diamond'),
    name='Non-Manifold (Bowtie) Vertices'
)

fig = go.Figure(data=[mesh_plot, wireframe, nm_points])
fig.update_layout(
    title="Swiss Cheese Sphere (Red = Bowtie Vertices)",
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False)
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()


### 5. MeshBVH and Self-Intersection
In this section, we create a completely random triangle soup, construct a `TriangleMesh`, and quickly identify all self-intersecting triangles directly using the mesh's built-in BVH methods.

In [12]:
import torch
import plotly.graph_objects as go
import conquer3d

# Create random triangle soup with both intersecting and non-intersecting triangles
N_soup = 20
torch.manual_seed(42)

# Generate random centers in a large space so most are isolated
centers = torch.rand((N_soup, 1, 3), dtype=torch.float32, device="cuda") * 3

# Force some triangles to be near each other to guarantee intersections
centers[1] = centers[0] + torch.rand((1, 1, 3), device="cuda") * 0.5
centers[3] = centers[2] + torch.rand((1, 1, 3), device="cuda") * 0.5
centers[5] = centers[4] + torch.rand((1, 1, 3), device="cuda") * 0.5

# Add small random offsets to create the 3 vertices of each triangle
offsets = torch.rand((N_soup, 3, 3), dtype=torch.float32, device="cuda") * 2
vertices_soup = (centers + offsets).view(N_soup * 3, 3)
triangles_soup = torch.arange(N_soup * 3, dtype=torch.int32, device="cuda").view(N_soup, 3)

# Create mesh
mesh_soup = conquer3d.data_structure.TriangleMesh(vertices_soup, triangles_soup)

# Check self intersection directly from mesh
has_intersection = mesh_soup.is_self_intersection()
intersecting_pairs = mesh_soup.get_self_intersection()

print(f"Has self intersection: {has_intersection}")
print(f"Found {intersecting_pairs.size(0)} intersecting pairs!")

# Visualize
fig = go.Figure()
v0 = vertices_soup[triangles_soup[:, 0]].cpu().numpy()
v1 = vertices_soup[triangles_soup[:, 1]].cpu().numpy()
v2 = vertices_soup[triangles_soup[:, 2]].cpu().numpy()

intersecting_indices = set()
if intersecting_pairs.size(0) > 0:
    intersecting_indices = set(intersecting_pairs.cpu().numpy().flatten())

for i in range(N_soup):
    x = [v0[i, 0], v1[i, 0], v2[i, 0], v0[i, 0]]
    y = [v0[i, 1], v1[i, 1], v2[i, 1], v0[i, 1]]
    z = [v0[i, 2], v1[i, 2], v2[i, 2], v0[i, 2]]
    
    is_intersecting = i in intersecting_indices
    color = 'red' if is_intersecting else 'blue'
    width = 5 if is_intersecting else 2
    name = f"Intersecting Triangle {i}" if is_intersecting else f"Triangle {i}"
    
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color=color, width=width), name=name))

fig.update_layout(scene=dict(aspectmode='data'), title="MeshBVH Self Intersection on Triangle Soup")
fig.show()

Has self intersection: True
Found 5 intersecting pairs!
